In [1]:
import numpy as np
from pulp import LpProblem, LpVariable, lpSum, LpMinimize, LpInteger, LpBinary, LpStatus

## Problem 1: Ham Deng

In [2]:
# constants
CAMPS = 6
c = [1_000, 10_000, 10_000, 25_000, 50_000, 600_000]
e = [1_000, 50_000, 20_000, 150_000, 50_000, 500_000]
d = [1, 1, 1, 15, 5, 2]
M = 15 # this is enough for the worse case (b/c x_5 is at most 6 and x_6 is at_most 15 so we pick M = 15)

In [3]:
lp = LpProblem("Ham_Deng", sense=LpMinimize)

x = [LpVariable(f"x_{i+1}", cat=LpInteger, lowBound=0) for i in range(CAMPS)]
m = LpVariable("m", cat=LpBinary)
# objective
lp += lpSum(c[i] * x[i] for i in range(CAMPS)), "Z"

# constraints
lp += lpSum(e[i] * x[i] for i in range(CAMPS)) >= 1_000_000, "At least 1 Million people"

for i in range(CAMPS):
    lp += d[i] * x[i] <= 30, f"Must have 1M in 30 days (can run simultaneously) (Campaign {i+1})"

lp  += x[5] <= M * (1-m), "Choose 6 can't choose 5"
lp  += x[4] <= M * m, "Choose 5 can't choose 6"

In [4]:
lp.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/wh/91lwmxs51lb2njldckd6jd440000gp/T/4932c6ec8f284cf2a7ad1814e5b56da0-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/wh/91lwmxs51lb2njldckd6jd440000gp/T/4932c6ec8f284cf2a7ad1814e5b56da0-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 14 COLUMNS
At line 51 RHS
At line 61 BOUNDS
At line 69 ENDATA
Problem MODEL has 9 rows, 7 columns and 16 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 190000 - 0.00 seconds
Cgl0004I processed model has 3 rows, 7 columns (7 integer (1 of which binary)) and 10 elements
Cutoff increment increased from 1e-05 to 1000
Cbc0012I Integer solution of 190000 found by DiveCoefficient after 0 iterations and 0 nodes (0.02 seconds)
Cbc

1

In [5]:
print("Status:", LpStatus[lp.status])

print("Min Cost = ", lp.objective.value())
for v in lp.variables():
    print(v.name, "=", v.varValue)

Status: Optimal
Min Cost =  190000.0
m = 0.0
x_1 = 0.0
x_2 = 14.0
x_3 = 0.0
x_4 = 2.0
x_5 = 0.0
x_6 = 0.0


## Problem 2: The Buff Boss

In [6]:
# constants
INGS = 4
w = [30, 35, 40, 20]
p = [24, 21, 6, 9]
c = [0, 0, 80, 50]

DAYS = 5
DAY_CAN_BUY = [0, 2] # monday and wednesday can go to shoppping
shelf_life = [1, 3, 3, 2]

consume_interval = [
    set(j for now in DAY_CAN_BUY for j in range(now, now+day) ) for day in shelf_life
]
print(consume_interval) # The day that can consume each ingredient

b = [[1 if j in interval else 0 for j in range(DAYS)] for interval in consume_interval]

print(*b, sep='\n')

[{0, 2}, {0, 1, 2, 3, 4}, {0, 1, 2, 3, 4}, {0, 1, 2, 3}]
[1, 0, 1, 0, 0]
[1, 1, 1, 1, 1]
[1, 1, 1, 1, 1]
[1, 1, 1, 1, 0]


In [7]:
lp = LpProblem("The_Buff_Boss", sense=LpMinimize)

x = [ [LpVariable(f"x_{i+1},{j+1}", cat=LpInteger, lowBound=0) for j in range(DAYS)] for i in range(INGS) ]

# objective
lp += lpSum(w[i] * x[i][j] * b[i][j] for i in range(INGS) for j in range(DAYS)), "Z"

# constraints
for j in range(DAYS):
    lp += lpSum(p[i] * x[i][j] * b[i][j] for i in range(INGS)) >= 200, f"protein each day >= 200 (day {j+1})"

for j in range(DAYS):
    lp += lpSum(c[i] * x[i][j] * b[i][j] for i in range(INGS)) >= 150, f"carb each day >= 150 (day {j+1})"
lp

The_Buff_Boss:
MINIMIZE
30*x_1,1 + 30*x_1,3 + 35*x_2,1 + 35*x_2,2 + 35*x_2,3 + 35*x_2,4 + 35*x_2,5 + 40*x_3,1 + 40*x_3,2 + 40*x_3,3 + 40*x_3,4 + 40*x_3,5 + 20*x_4,1 + 20*x_4,2 + 20*x_4,3 + 20*x_4,4 + 0
SUBJECT TO
protein_each_day_>=_200_(day_1): 24 x_1,1 + 21 x_2,1 + 6 x_3,1 + 9 x_4,1
 >= 200

protein_each_day_>=_200_(day_2): 21 x_2,2 + 6 x_3,2 + 9 x_4,2 >= 200

protein_each_day_>=_200_(day_3): 24 x_1,3 + 21 x_2,3 + 6 x_3,3 + 9 x_4,3
 >= 200

protein_each_day_>=_200_(day_4): 21 x_2,4 + 6 x_3,4 + 9 x_4,4 >= 200

protein_each_day_>=_200_(day_5): 21 x_2,5 + 6 x_3,5 >= 200

carb_each_day_>=_150_(day_1): 80 x_3,1 + 50 x_4,1 >= 150

carb_each_day_>=_150_(day_2): 80 x_3,2 + 50 x_4,2 >= 150

carb_each_day_>=_150_(day_3): 80 x_3,3 + 50 x_4,3 >= 150

carb_each_day_>=_150_(day_4): 80 x_3,4 + 50 x_4,4 >= 150

carb_each_day_>=_150_(day_5): 80 x_3,5 >= 150

VARIABLES
0 <= x_1,1 Integer
0 <= x_1,3 Integer
0 <= x_2,1 Integer
0 <= x_2,2 Integer
0 <= x_2,3 Integer
0 <= x_2,4 Integer
0 <= x_2,5 Integer
0

In [8]:
lp.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/wh/91lwmxs51lb2njldckd6jd440000gp/T/e55893041a944eada012ccd7a113ae7b-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/wh/91lwmxs51lb2njldckd6jd440000gp/T/e55893041a944eada012ccd7a113ae7b-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 15 COLUMNS
At line 89 RHS
At line 100 BOUNDS
At line 117 ENDATA
Problem MODEL has 10 rows, 16 columns and 25 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1638.75 - 0.00 seconds
Cgl0003I 0 fixed, 16 tightened bounds, 0 strengthened rows, 0 substitutions
Cgl0004I processed model has 9 rows, 16 columns (16 integer (0 of which binary)) and 24 elements
Cutoff increment increased from 1e-05 to 4.9999
Cbc0012I Integer solution

1

In [9]:
print("Status:", LpStatus[lp.status])

print("Min Cost = ", lp.objective.value())
for v in lp.variables():
    print(v.name, "=", v.varValue)

Status: Optimal
Min Cost =  1695.0
x_1,1 = 7.0
x_1,3 = 7.0
x_2,1 = 0.0
x_2,2 = 8.0
x_2,3 = 0.0
x_2,4 = 8.0
x_2,5 = 9.0
x_3,1 = 0.0
x_3,2 = 0.0
x_3,3 = 0.0
x_3,4 = 0.0
x_3,5 = 2.0
x_4,1 = 4.0
x_4,2 = 4.0
x_4,3 = 4.0
x_4,4 = 4.0


## Problem 3: Deadline Driven Development

In [10]:
# constants
TASKS = 7  # correspond to i
DAYS  = 14 # correspond to j
h = [6, 4, 5, 3, 6, 2, 4] # shifted index

In [11]:
lp = LpProblem("Deadline_Driven_Development", sense=LpMinimize)

y = [[LpVariable(f"y_{i+1},{j+1}", cat=LpBinary) for j in range(DAYS)] for i in range(TASKS) ]
m = [LpVariable(f"m_{j+1}", cat=LpBinary) for j in range(DAYS+1)]

lp += lpSum(m), "Z"

for j in range(DAYS-1):
    lp += m[j] >= m[j+1], f"No Free day ({j+1})"

lp += lpSum(lpSum(y[i]) for i in range(TASKS)) == 7, "All task must be completed"

for j in range(DAYS):
    for i in range(TASKS):
        lp += m[j] >= y[i][j], f"if m{j+1} is picked there must be some y_i,{j+1} that's also picked (i={i+1})"

for j in range(DAYS):
    lp += lpSum(lpSum(y[i][j] * h[i]) for i in range(TASKS)) <= 10, f"Government Regulation (Day {j+1})"

for i in range(TASKS):
    lp += lpSum(y[i][j] for j in range(DAYS)) == 1, f"each task can be done once. (Task {i+1})"

for j in range(DAYS):
    lp += y[0][j] + y[1][j] <= 1, f"task 1 and 2 cannot be done on the same day (Day {j+1})"

for j in range(DAYS):
    lp += y[2][j] - y[3][j] == 0, f"task 3 and 4 must be done on the same day (Day {j+1})"

for j in range(DAYS):
    lp += lpSum(y[4][k] for k in range(j+1)) + lpSum(y[5][k] for k in range(j+1)) >= 2*y[6][j], f"Before do task 7 in {j+1}th day must do task 5&6 in {1} to {j+1}th day"

lp

Deadline_Driven_Development:
MINIMIZE
1*m_1 + 1*m_10 + 1*m_11 + 1*m_12 + 1*m_13 + 1*m_14 + 1*m_15 + 1*m_2 + 1*m_3 + 1*m_4 + 1*m_5 + 1*m_6 + 1*m_7 + 1*m_8 + 1*m_9 + 0
SUBJECT TO
No_Free_day_(1): m_1 - m_2 >= 0

No_Free_day_(2): m_2 - m_3 >= 0

No_Free_day_(3): m_3 - m_4 >= 0

No_Free_day_(4): m_4 - m_5 >= 0

No_Free_day_(5): m_5 - m_6 >= 0

No_Free_day_(6): m_6 - m_7 >= 0

No_Free_day_(7): m_7 - m_8 >= 0

No_Free_day_(8): m_8 - m_9 >= 0

No_Free_day_(9): - m_10 + m_9 >= 0

No_Free_day_(10): m_10 - m_11 >= 0

No_Free_day_(11): m_11 - m_12 >= 0

No_Free_day_(12): m_12 - m_13 >= 0

No_Free_day_(13): m_13 - m_14 >= 0

All_task_must_be_completed: y_1,1 + y_1,10 + y_1,11 + y_1,12 + y_1,13 + y_1,14
 + y_1,2 + y_1,3 + y_1,4 + y_1,5 + y_1,6 + y_1,7 + y_1,8 + y_1,9 + y_2,1
 + y_2,10 + y_2,11 + y_2,12 + y_2,13 + y_2,14 + y_2,2 + y_2,3 + y_2,4 + y_2,5
 + y_2,6 + y_2,7 + y_2,8 + y_2,9 + y_3,1 + y_3,10 + y_3,11 + y_3,12 + y_3,13
 + y_3,14 + y_3,2 + y_3,3 + y_3,4 + y_3,5 + y_3,6 + y_3,7 + y_3,8 + y_3,

In [12]:
lp.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/wh/91lwmxs51lb2njldckd6jd440000gp/T/f74a350dc074427eb1177ffcac50e1af-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/wh/91lwmxs51lb2njldckd6jd440000gp/T/f74a350dc074427eb1177ffcac50e1af-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 180 COLUMNS
At line 1218 RHS
At line 1394 BOUNDS
At line 1508 ENDATA
Problem MODEL has 175 rows, 113 columns and 796 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1 - 0.00 seconds
Cgl0003I 1 fixed, 0 tightened bounds, 108 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 86 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 63 strengthened rows, 0 substitutions
Cgl0003I 0 fixe

1

In [13]:
print("Status:", LpStatus[lp.status])

print("Min day = ", lp.objective.value())
for v in lp.variables():
    if v.varValue == 0.0:
        continue
    print(v.name, "=", v.varValue)

Status: Optimal
Min day =  3.0
m_1 = 1.0
m_2 = 1.0
m_3 = 1.0
y_1,3 = 1.0
y_2,2 = 1.0
y_3,1 = 1.0
y_4,1 = 1.0
y_5,2 = 1.0
y_6,1 = 1.0
y_7,3 = 1.0
